## 1. Imports y configuración

In [1]:
import pandas as pd
from faker import Faker
import uuid
import random
import os

fake = Faker('es_ES')

#He fijado estos números, para que todos obtengamos
#los mismos datos al ejecutar el notebook
random.seed(42)
Faker.seed(42)

#Carpeta de salida
OUTPUT_DIR = '../data'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Librerías cargadas correctamente")
print(f"Carpeta de salida: {os.path.abspath(OUTPUT_DIR)}")


Librerías cargadas correctamente
Carpeta de salida: c:\Users\pc\Documents\Team Challenge\data


## 2. Enumerados del modelo

Estos valores están alineados con las decisiones de diseño de la Data Modeler.  
Cualquier cambio en estos enumerados debe coordinarse con el equipo.


In [2]:
#Valores
ACQUISITION_CHANNELS = ['organic', 'paid_ads', 'social_media', 'referral']
COUNTRIES = ['Spain', 'France', 'Germany', 'Italy', 'Portugal', 'Netherlands']

CATEGORY_NAMES = [
    'Smartphones', 'Laptops', 'Tablets', 'Headphones',
    'Cameras', 'Smart Home', 'Wearables', 'Gaming',
    'Accessories', 'Monitors'
]

CATEGORY_DESCS = {
    'Smartphones':  'Teléfonos inteligentes de última generación',
    'Laptops':      'Ordenadores portátiles para trabajo y ocio',
    'Tablets':      'Tabletas para productividad y entretenimiento',
    'Headphones':   'Auriculares y accesorios de audio',
    'Cameras':      'Cámaras digitales y de acción',
    'Smart Home':   'Dispositivos para el hogar inteligente',
    'Wearables':    'Relojes inteligentes y pulseras de actividad',
    'Gaming':       'Videojuegos, consolas y periféricos',
    'Accessories':  'Accesorios y complementos electrónicos',
    'Monitors':     'Monitores y pantallas para escritorio',
}

#Marcas por categoría
BRANDS_BY_CATEGORY = {
    'Smartphones':  ['Apple', 'Samsung', 'Xiaomi', 'Google', 'OnePlus'],
    'Laptops':      ['Apple', 'Dell', 'Lenovo', 'HP', 'Asus'],
    'Tablets':      ['Apple', 'Samsung', 'Lenovo', 'Microsoft', 'Huawei'],
    'Headphones':   ['Sony', 'Bose', 'Jabra', 'Sennheiser', 'Apple'],
    'Cameras':      ['Canon', 'Nikon', 'Sony', 'GoPro', 'Fujifilm'],
    'Smart Home':   ['Philips', 'Google', 'Amazon', 'Xiaomi', 'Ikea'],
    'Wearables':    ['Apple', 'Garmin', 'Fitbit', 'Samsung', 'Xiaomi'],
    'Gaming':       ['Sony', 'Microsoft', 'Nintendo', 'Razer', 'Logitech'],
    'Accessories':  ['Anker', 'Belkin', 'Logitech', 'Ugreen', 'Baseus'],
    'Monitors':     ['LG', 'Samsung', 'Dell', 'Asus', 'BenQ'],
}

print("Categorías definidas")


Categorías definidas


## 3. Tabla `categories`

Sin FKs → se genera primero.  
Los `created_at` se distribuyen entre hace 4 y 2 años para que sean anteriores a los productos.


In [3]:
def generar_categorias() -> pd.DataFrame:
    categorias = []

    for nombre in CATEGORY_NAMES:
        categorias.append({
            'category_id': str(uuid.uuid4()),
            'name':        nombre,
            'description': CATEGORY_DESCS[nombre],
            #Rango amplio. Para que no sean todos del 2026
            'created_at':  fake.date_time_between(start_date='-4y', end_date='-2y'),
        })

    return pd.DataFrame(categorias)


df_categories = generar_categorias()
df_categories


,category_id,name,description,created_at
0,f0aa30af-6bf5-4fd7-857b-adf59a6a61b1,Smartphones,Teléfonos inteligentes de última generación,2023-09-10 17:24:57
1,79f11369-3754-4842-9357-fe4112a77337,Laptops,Ordenadores portátiles para trabajo y ocio,2022-07-27 12:59:52
2,0013a422-6fee-4f9b-a6b1-84fa25582da0,Tablets,Tabletas para productividad y entretenimiento,2022-05-21 11:52:14
3,c6866289-6e76-4750-b6bf-155d4bfc4bf9,Headphones,Auriculares y accesorios de audio,2023-11-29 01:08:22
4,829cca52-0a10-414a-8de5-20065db09633,Cameras,Cámaras digitales y de acción,2022-12-01 16:33:16
5,0a1d1e18-484e-4f9a-a8da-90de91a4372f,Smart Home,Dispositivos para el hogar inteligente,2022-11-08 06:45:05
6,51752150-867f-4490-9f99-58f58216f331,Wearables,Relojes inteligentes y pulseras de actividad,2022-10-22 10:35:10
7,49c314c8-f52d-4ebb-b533-7a0dc7648c1e,Gaming,"Videojuegos, consolas y periféricos",2022-08-18 10:48:42
8,97fbe2e7-e609-41ab-bd45-83e593b8ab14,Accessories,Accesorios y complementos electrónicos,2023-11-25 02:27:17
9,bbb75cdf-29a1-4222-92b1-981e9bbde34b,Monitors,Monitores y pantallas para escritorio,2022-07-20 16:17:05


## 4. Tabla `products`

FK → `categories.category_id`

**Decisiones de diseño:**
- `price` > `cost` siempre (margen 20–80%) — garantizado por construcción
- `price` refleja el precio **actual**; el precio de venta histórico va en `order_items.unit_price`
- Marca elegida según categoría del producto (coherencia de catálogo)


In [4]:
def generar_productos(df_cats: pd.DataFrame, n: int = 70) -> pd.DataFrame:
    productos = []

    #category_id - nombre, para elegir la marca correcta
    cat_id_to_name = dict(zip(df_cats['category_id'], df_cats['name']))

    for _ in range(n):
        cat_id   = random.choice(df_cats['category_id'].tolist())
        cat_name = cat_id_to_name[cat_id]
        brand    = random.choice(BRANDS_BY_CATEGORY[cat_name])

        #PVP siempre > coste: margen entre 20 % y 80 %
        coste  = round(random.uniform(20.0, 800.0), 2)
        precio = round(coste * random.uniform(1.2, 1.8), 2)

        #Nombre realista
        model_code = fake.bothify('??-###').upper()
        name = f"{brand} {cat_name[:-1] if cat_name.endswith('s') else cat_name} {model_code}"

        productos.append({
            'product_id':  str(uuid.uuid4()),
            'category_id': cat_id,
            'name':        name,
            'description': fake.sentence(nb_words=10),
            'brand':       brand,
            'price':       precio,
            'cost':        coste,
            'stock':       random.randint(0, 400),
            'is_active':   random.choice([True, True, True, False]),  # 75 % activos
            #Productos más recientes
            'created_at':  fake.date_time_between(start_date='-2y', end_date='-6m'),
        })

    return pd.DataFrame(productos)


df_products = generar_productos(df_categories, n=70)
df_products.head()


,product_id,category_id,name,description,brand,price,cost,stock,is_active,created_at
0,740b9f41-2845-45fb-9b4a-523dbfddf2a3,79f11369-3754-4842-9357-fe4112a77337,Apple Laptop BC-819,Sí lado clase importancia vivir habrá.,Apple,806.02,598.41,71,True,2025-03-23 09:34:21
1,ce7bf56b-b33d-478c-8c98-d284d1a26a2f,97fbe2e7-e609-41ab-bd45-83e593b8ab14,Anker Accessorie RZ-379,Peso medio relaciones alguna acuerdo serán.,Anker,585.86,480.58,47,True,2024-07-19 22:29:38
2,f5424f64-acbc-4b41-bb7e-c13de4a79279,c6866289-6e76-4750-b6bf-155d4bfc4bf9,Apple Headphone WW-161,Nombre había etc hecho material te luis pueda ...,Apple,752.35,489.57,366,False,2025-12-23 00:11:41
3,c677a52c-cb80-46c2-98b7-fc051fcf6b86,c6866289-6e76-4750-b6bf-155d4bfc4bf9,Sennheiser Headphone GY-413,Grupos período sea igual grande precisamente h...,Sennheiser,808.49,479.63,3,True,2025-06-20 10:14:43
4,7d4a2909-77a4-4146-af01-ca616179becc,51752150-867f-4490-9f99-58f58216f331,Fitbit Wearable YR-327,Hizo sé llega yo informe puerto estoy uno huma...,Fitbit,314.67,236.74,390,True,2026-03-19 16:50:44


## 5. Tabla `customers`

Sin FKs → se genera en paralelo a `categories`.

**Decisiones de diseño (justificación 3NF):**  
`country` se almacena como STRING directamente en `customers`. No se crea una tabla `countries` separada porque no hay otros atributos que dependan de `country` — esto es válido en 3NF.


In [5]:
def generar_clientes(n: int = 500) -> pd.DataFrame:
    clientes = []

    for _ in range(n):
        clientes.append({
            'customer_id':         str(uuid.uuid4()),
            'first_name':          fake.first_name(),
            'last_name':           fake.last_name(),
            'email':               fake.unique.email(), 
            'phone':               fake.phone_number(),
            'city':                fake.city(),
            'country':             random.choice(COUNTRIES),
            'acquisition_channel': random.choice(ACQUISITION_CHANNELS),
            'registered_at':       fake.date_time_between(start_date='-2y', end_date='now'),
            'is_active':           random.choice([True, True, True, False]),  #75% activos
        })

    return pd.DataFrame(clientes)


df_customers = generar_clientes(500)
df_customers.head()


,customer_id,first_name,last_name,email,phone,city,country,acquisition_channel,registered_at,is_active
0,ce6b66e8-2316-4e15-8e1b-7d18fcc6cf12,Ciriaco,Garay,ayusomiriam@example.com,+34880 293 183,Ceuta,Portugal,organic,2025-01-11 17:57:01,False
1,67d4bf54-5ff3-4b9b-b843-da7d2f7a8d71,María Ángeles,Coloma,lluchrosa-maria@example.org,+34 842842102,Huelva,Portugal,paid_ads,2026-01-05 03:32:16,True
2,c08bfa27-14a6-45a1-b327-e9f715217005,Modesta,Gárate,alcides24@example.org,+34841 681 177,Málaga,Spain,referral,2025-08-05 15:53:35,True
3,9afaaf53-5b40-4afa-847d-6536001f4f25,César,Sans,encarnacionaroca@example.net,+34984 47 00 76,Soria,Portugal,paid_ads,2024-07-24 09:42:38,True
4,e859e3c0-de4a-4645-a8f2-ba6c46ed76eb,Amor,Salcedo,berrocalmodesto@example.net,+34 845124998,Jaén,Italy,organic,2025-08-08 13:20:51,True


## 6. Verificaciones de calidad

Comprobaciones rápidas antes de exportar para detectar errores antes de cargar en BigQuery.


In [6]:
print("=" * 50)
print("RESUMEN DE TABLAS GENERADAS")
print("=" * 50)
print(f"  categories : {len(df_categories):>5} filas")
print(f"  products   : {len(df_products):>5} filas")
print(f"  customers  : {len(df_customers):>5} filas")

print()
print("── Tipos de dato ──")
print("\ncategories:")
print(df_categories.dtypes)
print("\nproducts:")
print(df_products.dtypes)
print("\ncustomers:")
print(df_customers.dtypes)


RESUMEN DE TABLAS GENERADAS
  categories :    10 filas
  products   :    70 filas
  customers  :   500 filas

── Tipos de dato ──

categories:
category_id               str
name                      str
description               str
created_at     datetime64[us]
dtype: object

products:
product_id                str
category_id               str
name                      str
description               str
brand                     str
price                 float64
cost                  float64
stock                   int64
is_active                bool
created_at     datetime64[us]
dtype: object

customers:
customer_id                       str
first_name                        str
last_name                         str
email                             str
phone                             str
city                              str
country                           str
acquisition_channel               str
registered_at          datetime64[us]
is_active                        bool
dtype:

In [9]:
#Precio siempre mayor que costes
assert (df_products['price'] > df_products['cost']).all()
print("price > cost en todos los productos")

#Emails no repes
assert df_customers['email'].nunique() == len(df_customers)
print("Todos los emails están sin duplicar")

#Comprobar que category_id de products existen en categories
ids_validos = set(df_categories['category_id'])
assert df_products['category_id'].isin(ids_validos).all()
print("Categoría correcta")

#Mínimos que se esperan(enunciado)
assert len(df_categories) == 10
assert len(df_products)   >= 70
assert len(df_customers)  >= 500
print("Mínimos del enunciado cumplidos")

print()
print("Todas las verificaciones han sido superadas - listos para exportar")


price > cost en todos los productos
Todos los emails están sin duplicar
Categoría correcta
Mínimos del enunciado cumplidos

Todas las verificaciones han sido superadas - listos para exportar


## 7. Exportar a CSV

Los CSVs se guardan en `data/` para que el notebook `01_setup_bigquery.ipynb`  
pueda cargarlos en BigQuery.


In [10]:
df_categories.to_csv(f'{OUTPUT_DIR}/categories.csv', index=False)
df_products.to_csv(f'{OUTPUT_DIR}/products.csv',     index=False)
df_customers.to_csv(f'{OUTPUT_DIR}/customers.csv',   index=False)

print("Archivos exportados:")
for fname in ['categories.csv', 'products.csv', 'customers.csv']:
    path = f'{OUTPUT_DIR}/{fname}'
    rows = len(pd.read_csv(path))
    print(f"{fname:<25} → {rows:>5} filas")


Archivos exportados:
categories.csv            →    10 filas
products.csv              →    70 filas
customers.csv             →   500 filas
